# RelCheck — Full 600-Image Run (Together.ai)

## What this notebook computes
- **B1**: BLIP-2 original captions (no correction)
- **B2**: Self-refinement (Llama refines without visual evidence)
- **B3**: Blind correction (Llama corrects with no KB)
- **RelCheck v2**: Full pipeline (GDINO + geometry + Maverick + structured rewrite)

## Tables produced
- **Table 1**: R-POPE LLM-Judge (main metric) — B1 / B2 / B3 / RelCheck v2
- **Table 3**: KB Source Ablation — no KB / objects only / objects+geometry / full KB
- **Table 4**: Correction Method Ablation — blind / KB dump / structured (RelCheck v2)
- **Table 5**: CLIPScore — all methods
- **Table 7**: Filtered R-POPE — corrected images only
- **Table 8**: Per-relation-type breakdown
- **Table 9**: Pipeline statistics

## Checkpoint / resume
Every expensive stage saves to Drive. If Colab disconnects, re-run from Cell 0 — completed stages load from cache automatically.

## Runtime estimate (T4 GPU, Together.ai)
- Cell 3 (BLIP-2 captions): ~10 min
- Cell 4 (KB construction): ~25 min
- Cell 5 (RelCheck enrichment): ~20 min
- Cell 6 (ablation correctors): ~30 min
- Cell 7 (R-POPE eval): ~45 min
- Cells 8-10 (metrics + tables + figures): ~10 min
- **Total: ~2.5 hrs**

In [ ]:
# ============================================================
# Cell 0 — Install + Setup
# ============================================================
!pip install -q together transformers pillow torch torchvision
!pip install -q spacy open-clip-torch
!python -m spacy download en_core_web_sm -q

import os, json, base64, re, time, random, csv
from io import BytesIO
from collections import defaultdict, Counter
from pathlib import Path
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from PIL import Image
import torch
from together import Together
from transformers import (
    AutoProcessor, AutoModelForZeroShotObjectDetection,
    Blip2Processor, Blip2ForConditionalGeneration,
)
import spacy

# ── Settings ──────────────────────────────────────────────
TOGETHER_API_KEY = ""   # <-- paste your Together.ai key
N_IMAGES         = 600  # set to None to use all available
RANDOM_SEED      = 42

# Drive paths
DRIVE_IMAGES_DIR = "/content/drive/MyDrive/RelCheck_Data/images"
RBENCH_PATH      = "/content/drive/MyDrive/RelCheck_Data/rbench_data.json"
SAVE_DIR         = "/content/drive/MyDrive/RelCheck_Data/run_600"
FIGS_DIR         = f"{SAVE_DIR}/figures"

# Model IDs
VLM_MODEL = "meta-llama/Llama-4-Maverick-17B-128E-Instruct-FP8"
LLM_MODEL = "meta-llama/Llama-3.3-70B-Instruct-Turbo"
GDINO_ID  = "IDEA-Research/grounding-dino-tiny"
BLIP2_ID  = "Salesforce/blip2-flan-t5-xl"

# Detection
DETECTION_THRESHOLD = 0.25

# ── Init ──────────────────────────────────────────────────
os.environ["TOGETHER_API_KEY"] = TOGETHER_API_KEY
client = Together(api_key=TOGETHER_API_KEY)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
random.seed(RANDOM_SEED)

os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(FIGS_DIR, exist_ok=True)

nlp = spacy.load("en_core_web_sm")

# ── Helper: safe LLM call with retry ──────────────────────
def llm_call(messages, model=LLM_MODEL, max_tokens=600, retries=3):
    for attempt in range(retries):
        try:
            resp = client.chat.completions.create(
                model=model, messages=messages,
                temperature=0.0, max_tokens=max_tokens,
            )
            return resp.choices[0].message.content.strip()
        except Exception as e:
            if attempt < retries - 1:
                wait = 2 ** attempt
                print(f"  API error (attempt {attempt+1}): {e} — retrying in {wait}s")
                time.sleep(wait)
            else:
                return None

# ── Helper: incremental save ───────────────────────────────
def save_checkpoint(data, path):
    with open(path, "w") as f:
        json.dump(data, f, indent=2, default=str)

def load_checkpoint(path):
    if os.path.exists(path):
        with open(path) as f:
            return json.load(f)
    return None

print(f"Device:  {DEVICE}")
print(f"Save to: {SAVE_DIR}")
print(f"Images:  {N_IMAGES if N_IMAGES else 'all available'}")

In [ ]:
# ============================================================
# Cell 1 — Load Models (GDINO + BLIP-2)
# ============================================================
from google.colab import drive
if not os.path.exists("/content/drive/MyDrive"):
    drive.mount("/content/drive")

print("Loading GroundingDINO...")
gdino_processor = AutoProcessor.from_pretrained(GDINO_ID)
gdino_model = AutoModelForZeroShotObjectDetection.from_pretrained(GDINO_ID).to(DEVICE)
print("  GroundingDINO loaded.")

print("Loading BLIP-2...")
blip2_processor = Blip2Processor.from_pretrained(BLIP2_ID)
blip2_model = Blip2ForConditionalGeneration.from_pretrained(
    BLIP2_ID, torch_dtype=torch.float16, device_map="auto"
)
print(f"  BLIP-2 loaded.")
print(f"\nReady on {DEVICE}.")

In [ ]:
# ============================================================
# Cell 2 — Load Images + R-Bench Data
# ============================================================

# Load R-Bench
with open(RBENCH_PATH) as f:
    rbench_data = json.load(f)
print(f"R-Bench: {len(rbench_data)} questions total")

rbench_by_image = defaultdict(list)
for entry in rbench_data:
    rbench_by_image[entry['image']].append(entry)
print(f"R-Bench unique images: {len(rbench_by_image)}")

# Find images on Drive that have R-Bench questions
cached_images = list(Path(DRIVE_IMAGES_DIR).glob("*.jpg")) + \
                list(Path(DRIVE_IMAGES_DIR).glob("*.jpeg"))
cached_stems = {p.stem: p for p in cached_images}

matched = {}
for rb_key in rbench_by_image:
    stem = re.sub(r'\.(jpg|jpeg|png)$', '', rb_key)
    if stem in cached_stems:
        matched[stem] = {
            "path": str(cached_stems[stem]),
            "questions": rbench_by_image[rb_key],
        }

print(f"Images on Drive with R-Bench questions: {len(matched)}")

# Sample N_IMAGES (or use all)
all_ids = list(matched.keys())
if N_IMAGES and len(all_ids) > N_IMAGES:
    selected_ids = random.sample(all_ids, N_IMAGES)
    print(f"Sampled {N_IMAGES} of {len(all_ids)} available images")
else:
    selected_ids = all_ids
    print(f"Using all {len(selected_ids)} available images")

# Load images into memory + question mapping
images = {}
img_to_questions = {}
for img_id in selected_ids:
    info = matched[img_id]
    images[img_id] = Image.open(info["path"]).convert("RGB")
    img_to_questions[img_id] = info["questions"]

total_q = sum(len(qs) for qs in img_to_questions.values())
print(f"\nLoaded {len(images)} images")
print(f"R-Bench questions: {total_q} ({total_q/len(images):.1f} per image)")

In [ ]:
# ============================================================
# Cell 3 — Stage 1: Caption Generation (BLIP-2)
# ============================================================
# Checkpoint: loads from Drive if already computed.

BLIP2_PROMPT = "Describe the relationships between objects and people in this image."
CAPTIONS_PATH = f"{SAVE_DIR}/captions.json"

def caption_with_blip2(image):
    inputs = blip2_processor(
        images=image, text=BLIP2_PROMPT, return_tensors="pt"
    ).to(blip2_model.device, torch.float16)
    with torch.no_grad():
        out = blip2_model.generate(**inputs, max_new_tokens=80)
    return blip2_processor.decode(out[0], skip_special_tokens=True).strip()

# ── Load or compute ──
captions = load_checkpoint(CAPTIONS_PATH) or {}

if len(captions) == len(images):
    print(f"Loaded {len(captions)} cached captions from Drive.")
else:
    todo = [img_id for img_id in images if img_id not in captions]
    print(f"Captioning {len(todo)} images (have {len(captions)} cached)...")

    for i, img_id in enumerate(todo):
        captions[img_id] = caption_with_blip2(images[img_id])
        if (i + 1) % 50 == 0 or i == 0:
            print(f"  [{i+1}/{len(todo)}] {img_id[:20]}: {captions[img_id][:70]}")
            save_checkpoint(captions, CAPTIONS_PATH)  # incremental save

    save_checkpoint(captions, CAPTIONS_PATH)
    print(f"\nDone. Captions saved to Drive.")

# Quick sanity check
lens = [len(c) for c in captions.values()]
print(f"Caption length: min={min(lens)}, mean={sum(lens)//len(lens)}, max={max(lens)} chars")

In [ ]:
# ============================================================
# Cell 4 — Stage 2: Visual Knowledge Base Construction
# ============================================================
# Three layers per image:
#   HARD  — GroundingDINO: objects + counts + bboxes (deterministic)
#   GEOM  — Bbox geometry: pairwise spatial relationships (deterministic)
#   SOFT  — Maverick VLM: actions, attributes, relationships (visual)
# Checkpoint: loads from Drive if already computed.

KB_PATH = f"{SAVE_DIR}/knowledge_bases.json"

BROAD_CATEGORIES = [
    "person", "man", "woman", "child", "boy", "girl",
    "dog", "cat", "bird", "horse", "animal",
    "car", "bicycle", "motorcycle", "bus", "truck",
    "chair", "table", "bench", "couch", "bed",
    "food", "plate", "bowl", "cup", "bottle", "glass",
    "bag", "umbrella", "phone", "book", "sign",
    "hat", "jacket", "vest", "helmet", "glasses",
    "tree", "flower", "grass", "water",
]

MAVERICK_KB_PROMPT = """Describe the RELATIONSHIPS between objects in this image.

An object detector found these objects:
{detection_list}

For each pair of detected objects that interact, describe:
- SPATIAL: Where are they relative to each other?
- ACTIONS: What is each person/animal doing?
- ATTRIBUTES: Relevant attributes tied to relationships

Rules: only describe what you can clearly see. Be brief and factual.
Format as a numbered list."""


def _clean_label(label):
    label = label.strip().lower()
    words = label.split()
    while words and words[0] in ('a', 'an', 'the'):
        words = words[1:]
    seen, clean = [], []
    for w in words:
        if w not in seen:
            seen.append(w)
            clean.append(w)
    return ' '.join(clean) if clean else label


def detect_objects(image, queries):
    text = ". ".join(queries) + "."
    inputs = gdino_processor(images=image, text=text, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        outputs = gdino_model(**inputs)
    results = gdino_processor.post_process_grounded_object_detection(
        outputs, inputs.input_ids,
        threshold=DETECTION_THRESHOLD, text_threshold=DETECTION_THRESHOLD,
        target_sizes=[image.size[::-1]],
    )[0]
    W, H = image.size
    lkey = "text_labels" if "text_labels" in results else "labels"
    dets = []
    for score, label, box in zip(results["scores"], results[lkey], results["boxes"]):
        x1, y1, x2, y2 = box.tolist()
        dets.append((_clean_label(label), score.item(), [x1/W, y1/H, x2/W, y2/H]))
    return dets


def dedup(dets):
    seen = {}
    for label, score, bbox in sorted(dets, key=lambda x: -x[1]):
        key = label.lower().strip()
        if key not in seen:
            seen[key] = [(score, bbox)]
        else:
            is_dup = any(
                max(0, min(bbox[2], eb[2]) - max(bbox[0], eb[0])) *
                max(0, min(bbox[3], eb[3]) - max(bbox[1], eb[1])) /
                ((bbox[2]-bbox[0])*(bbox[3]-bbox[1]) +
                 (eb[2]-eb[0])*(eb[3]-eb[1]) -
                 max(0, min(bbox[2], eb[2])-max(bbox[0], eb[0])) *
                 max(0, min(bbox[3], eb[3])-max(bbox[1], eb[1])) + 1e-8) > 0.5
                for _, eb in seen[key]
            )
            if not is_dup:
                seen[key].append((score, bbox))
    return [(label, s, b) for label, entries in seen.items() for s, b in entries]


def compute_spatial_facts(dets):
    facts = []
    counts = Counter(l for l, _, _ in dets)
    for l, c in counts.items():
        facts.append(f"There {'are' if c > 1 else 'is'} {c} '{l}'")
    labels = list(counts.keys())
    for i, l1 in enumerate(labels):
        for j, l2 in enumerate(labels):
            if i >= j:
                continue
            _, b1 = max([(s, b) for l, s, b in dets if l == l1], key=lambda x: x[0])
            _, b2 = max([(s, b) for l, s, b in dets if l == l2], key=lambda x: x[0])
            c1x, c1y = (b1[0]+b1[2])/2, (b1[1]+b1[3])/2
            c2x, c2y = (b2[0]+b2[2])/2, (b2[1]+b2[3])/2
            contained = (b1[0]>=b2[0]-0.05 and b1[2]<=b2[2]+0.05 and
                         b1[1]>=b2[1]-0.05 and b1[3]<=b2[3]+0.05)
            if contained:
                facts.append(f"'{l1}' is on/inside '{l2}'")
                continue
            dx, dy = c1x-c2x, c1y-c2y
            pos = ("to the left of" if dx < 0 else "to the right of") if abs(dx) > abs(dy) \
                  else ("above" if dy < 0 else "below")
            facts.append(f"'{l1}' is {pos} '{l2}'")
    return facts


def encode_b64(image):
    buf = BytesIO()
    image.save(buf, format="JPEG", quality=85)
    return base64.b64encode(buf.getvalue()).decode("utf-8")


def extract_nouns(caption):
    doc = nlp(caption)
    nouns = set()
    for chunk in doc.noun_chunks:
        nouns.add(chunk.root.text.lower())
        nouns.add(chunk.text.lower().strip())
    return list(nouns)


# ── Load or compute KBs ──
# KB stored as serializable dict (detections as lists, not tuples)
kb_raw = load_checkpoint(KB_PATH) or {}

# Reconstruct detections as tuples if loaded from JSON
knowledge_bases = {}
for img_id, kb in kb_raw.items():
    knowledge_bases[img_id] = {
        "hard_facts": kb["hard_facts"],
        "spatial_facts": kb["spatial_facts"],
        "visual_description": kb["visual_description"],
        "detections": [tuple(d) for d in kb.get("detections", [])],
    }

if len(knowledge_bases) == len(images):
    print(f"Loaded {len(knowledge_bases)} cached KBs from Drive.")
else:
    todo = [img_id for img_id in images if img_id not in knowledge_bases]
    print(f"Building KB for {len(todo)} images (have {len(knowledge_bases)} cached)...")

    for idx, img_id in enumerate(todo):
        img = images[img_id]
        cap = captions.get(img_id, "")
        kb = {"hard_facts": [], "spatial_facts": [], "visual_description": "", "detections": []}

        # Detection
        queries = list(set(extract_nouns(cap) + BROAD_CATEGORIES))
        raw = detect_objects(img, queries)
        dets = dedup(raw)
        kb["detections"] = dets

        counts = Counter(l for l, _, _ in dets)
        det_str = "".join(f"- {l} ({c}x)\n" for l, c in counts.most_common())
        kb["hard_facts"] = [f"{c}x {l}" for l, c in counts.most_common()]
        kb["spatial_facts"] = compute_spatial_facts(dets)

        # Maverick VLM
        b64 = encode_b64(img)
        resp_text = llm_call(
            messages=[{"role": "user", "content": [
                {"type": "text", "text": MAVERICK_KB_PROMPT.format(detection_list=det_str)},
                {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{b64}"}},
            ]}],
            model=VLM_MODEL, max_tokens=500,
        )
        kb["visual_description"] = resp_text or ""

        knowledge_bases[img_id] = kb

        if (idx + 1) % 50 == 0 or idx == 0:
            print(f"  [{idx+1}/{len(todo)}] {img_id[:15]}: {len(dets)} objs, "
                  f"{len(kb['spatial_facts'])} spatial facts")
            # Incremental save (convert tuples to lists for JSON)
            save_kb = {k: {**v, "detections": [list(d) for d in v["detections"]]}
                       for k, v in knowledge_bases.items()}
            save_checkpoint(save_kb, KB_PATH)
        time.sleep(0.3)

    save_kb = {k: {**v, "detections": [list(d) for d in v["detections"]]}
               for k, v in knowledge_bases.items()}
    save_checkpoint(save_kb, KB_PATH)
    print(f"\nDone. KBs saved to Drive.")

avg_objs = np.mean([len(Counter(l for l,_,_ in kb["detections"])) for kb in knowledge_bases.values()])
avg_spatial = np.mean([len(kb["spatial_facts"]) for kb in knowledge_bases.values()])
print(f"Avg objects/image: {avg_objs:.1f} | Avg spatial facts: {avg_spatial:.1f}")

In [ ]:
# ============================================================
# Cell 5 — Stage 3: Full RelCheck Enrichment
# ============================================================
# Produces: enriched captions (RelCheck v2) with KB-grounded rewrite
# Also logs: errors_found, missing_found, edit_rate, relation_type stats
# Checkpoint: loads from Drive if already computed.

ENRICHED_PATH = f"{SAVE_DIR}/enriched.json"

ANALYSIS_PROMPT = """You are a caption quality improver. You have an image caption and a Visual Knowledge Base (KB).

CAPTION: "{caption}"

=== VISUAL KNOWLEDGE BASE ===
DETECTED OBJECTS (highly reliable):
{hard_facts}

SPATIAL RELATIONSHIPS (from bounding box geometry — deterministic):
{spatial_facts}

VISUAL DESCRIPTION (from vision model):
{visual_description}

=== TASK ===
Step 1: Find problems:
  - ERRORS: Caption claims that KB contradicts
  - MISSING: Important KB facts absent from caption

Step 2: Write IMPROVED caption (2-3 sentences):
  - Fix errors using KB evidence
  - Add most important missing facts
  - Keep everything correct from original
  - Only include KB-supported facts

Output valid JSON:
{{"errors": [{{"claim": "...", "correction": "..."}}],
  "missing": [{{"fact": "...", "source": "..."}}],
  "improved_caption": "The rewritten caption."}}"""

VERIFY_PROMPT = """Check if this caption is accurate based on KB evidence.

Caption: "{rewritten}"
KB objects: {objects}
KB relationships: {relationships}

FAIL only if caption contradicts KB, has bad grammar, or has nonsensical repetition.
Answer ONLY "PASS" or "FAIL: [reason]"."""


def levenshtein(s1, s2):
    if len(s1) < len(s2): return levenshtein(s2, s1)
    if not s2: return len(s1)
    prev = range(len(s2)+1)
    for c1 in s1:
        curr = [prev[0]+1]
        for j, c2 in enumerate(s2):
            curr.append(min(curr[-1]+1, prev[j+1]+1, prev[j]+(c1!=c2)))
        prev = curr
    return prev[-1]


def enrich_caption(img_id, caption, kb):
    """Run full RelCheck enrichment. Returns dict with corrected caption + metadata."""
    hard = "\n".join(f"- {f}" for f in kb["hard_facts"]) or "- None detected"
    spatial = "\n".join(f"- {f}" for f in kb["spatial_facts"]) or "- No spatial facts"
    visual = kb["visual_description"][:400] or "- No visual description"

    prompt = ANALYSIS_PROMPT.format(
        caption=caption, hard_facts=hard, spatial_facts=spatial, visual_description=visual
    )
    improved = caption
    errors, missing = [], []

    raw = llm_call([{"role": "user", "content": prompt}], max_tokens=800)
    if raw:
        raw = re.sub(r'^```json\s*', '', raw)
        raw = re.sub(r'```\s*$', '', raw)
        if raw.count('{') > raw.count('}'):
            raw += '"}'
        try:
            result = json.loads(raw)
            errors = result.get("errors", [])
            missing = result.get("missing", [])
            if errors or missing:
                cand = result.get("improved_caption", "").strip().strip('"').strip("'")
                if cand and len(cand) >= 10:
                    improved = cand
        except Exception:
            pass

    # Sentence count check
    if improved != caption:
        n_sent = len([s for s in re.split(r'[.!?]+', improved) if s.strip()])
        if n_sent > 4:
            improved = caption

    # Verification
    if improved != caption:
        counts = Counter(l for l, _, _ in kb.get("detections", []))
        obj_str = ", ".join(f"{c}x {l}" for l, c in counts.most_common(8))
        rel_str = kb["visual_description"][:300]
        verdict = llm_call(
            [{"role": "user", "content": VERIFY_PROMPT.format(
                rewritten=improved, objects=obj_str, relationships=rel_str
            )}],
            max_tokens=50,
        )
        if verdict and verdict.upper().startswith("FAIL"):
            improved = caption

    edit_rate = levenshtein(caption, improved) / max(len(caption), len(improved), 1)

    return {
        "caption": caption,
        "corrected": improved,
        "errors": errors,
        "missing": missing,
        "edit_rate": edit_rate,
        "status": "modified" if improved != caption else "unchanged",
    }


# ── Load or compute enriched captions ──
enriched = load_checkpoint(ENRICHED_PATH) or {}

if len(enriched) == len(images):
    print(f"Loaded {len(enriched)} cached enriched captions from Drive.")
else:
    todo = [img_id for img_id in images if img_id not in enriched]
    print(f"Enriching {len(todo)} images (have {len(enriched)} cached)...")

    for idx, img_id in enumerate(todo):
        enriched[img_id] = enrich_caption(
            img_id, captions[img_id], knowledge_bases[img_id]
        )
        if (idx + 1) % 50 == 0 or idx == 0:
            n_mod = sum(1 for r in enriched.values() if r["status"] == "modified")
            print(f"  [{idx+1}/{len(todo)}] modified so far: {n_mod}")
            save_checkpoint(enriched, ENRICHED_PATH)
        time.sleep(0.3)

    save_checkpoint(enriched, ENRICHED_PATH)
    print("Done. Enriched captions saved to Drive.")

n_mod = sum(1 for r in enriched.values() if r["status"] == "modified")
n_err = sum(len(r["errors"]) for r in enriched.values())
n_miss = sum(len(r["missing"]) for r in enriched.values())
avg_edit = np.mean([r["edit_rate"] for r in enriched.values() if r["status"] == "modified"]) if n_mod else 0
print(f"\nEnrichment summary:")
print(f"  Images modified: {n_mod}/{len(enriched)} ({n_mod/len(enriched):.0%})")
print(f"  Errors found: {n_err} | Missing facts: {n_miss}")
print(f"  Avg edit rate (modified): {avg_edit:.0%}")

In [ ]:
# ============================================================
# Cell 6 — Ablation Correctors
# ============================================================
# Runs post-hoc on saved captions + KBs — no image re-processing.
#
# TABLE 3 (KB Source Ablation) — varies what evidence we give:
#   b3     : No KB at all
#   kb_obj : Objects only (GDINO counts, no geometry, no VLM)
#   kb_geom: Objects + geometric spatial relations (no VLM)
#   full   : Full KB (= RelCheck v2, already in Cell 5)
#
# TABLE 4 (Correction Method Ablation) — varies how we correct:
#   b2      : Self-refinement (Llama refines without any visual evidence)
#   b3      : Blind correction (same as Table 3 b3, shared)
#   kb_dump : Full KB as raw unstructured text, free rewrite
#   full    : Structured analysis → targeted fix (= RelCheck v2)
#
# Checkpoint: loads from Drive if already computed.

ABLATIONS_PATH = f"{SAVE_DIR}/ablations.json"

# ── Prompt templates ──────────────────────────────────────

B2_PROMPT = """This image caption may contain inaccuracies about object relationships.
Please review it carefully and rewrite it to be more accurate and complete.
Keep the same style and length. Only fix what seems wrong.

Caption: "{caption}"

Rewritten caption (one paragraph, 1-3 sentences):"""

B3_PROMPT = """This image caption may contain relational hallucinations (incorrect claims about
how objects relate to each other). Please rewrite it to fix any errors.

Caption: "{caption}"

Corrected caption (1-3 sentences, same style):"""

KB_OBJ_PROMPT = """Improve this image caption using the list of detected objects.

CAPTION: "{caption}"

DETECTED OBJECTS:
{obj_facts}

Rewrite the caption to fix any errors based on the detected objects and add important missing objects.
Keep it to 2-3 natural sentences.

Improved caption:"""

KB_GEOM_PROMPT = """Improve this image caption using detected objects and their spatial positions.

CAPTION: "{caption}"

DETECTED OBJECTS:
{obj_facts}

SPATIAL RELATIONSHIPS (from bounding boxes — deterministic):
{spatial_facts}

Rewrite the caption to fix spatial errors and add missing spatial details.
Keep it to 2-3 natural sentences.

Improved caption:"""

KB_DUMP_PROMPT = """Improve this image caption using all available visual evidence.

CAPTION: "{caption}"

FULL VISUAL KNOWLEDGE BASE:
Objects: {obj_facts}
Spatial: {spatial_facts}
Visual description: {visual_desc}

Rewrite the caption to be as accurate and complete as possible. 2-3 natural sentences.

Improved caption:"""


def run_ablation_corrector(prompt_fn, images_dict, captions_dict, kb_dict,
                           variant_name, cached={}):
    """Run a correction variant for all images. Returns {img_id: corrected_caption}."""
    results = dict(cached)  # start from cached
    todo = [img_id for img_id in images_dict if img_id not in results]

    if not todo:
        print(f"  {variant_name}: all {len(results)} cached.")
        return results

    print(f"  {variant_name}: correcting {len(todo)} images...")
    for idx, img_id in enumerate(todo):
        caption = captions_dict[img_id]
        kb = kb_dict[img_id]
        prompt = prompt_fn(caption, kb)

        raw = llm_call([{"role": "user", "content": prompt}], max_tokens=300)
        corrected = caption  # default: unchanged
        if raw:
            raw = raw.strip().strip('"').strip("'")
            if len(raw) >= 10:
                corrected = raw

        results[img_id] = corrected
        if (idx + 1) % 100 == 0:
            print(f"    [{idx+1}/{len(todo)}]")
        time.sleep(0.25)

    return results


# ── Load or compute ablations ──
ablations_raw = load_checkpoint(ABLATIONS_PATH) or {}

# B2: self-refinement
if "b2" not in ablations_raw:
    ablations_raw["b2"] = {}
ablations_raw["b2"] = run_ablation_corrector(
    lambda cap, kb: B2_PROMPT.format(caption=cap),
    images, captions, knowledge_bases, "B2 (self-refine)",
    cached=ablations_raw["b2"],
)
save_checkpoint(ablations_raw, ABLATIONS_PATH)

# B3: blind correction (no KB)
if "b3" not in ablations_raw:
    ablations_raw["b3"] = {}
ablations_raw["b3"] = run_ablation_corrector(
    lambda cap, kb: B3_PROMPT.format(caption=cap),
    images, captions, knowledge_bases, "B3 (blind)",
    cached=ablations_raw["b3"],
)
save_checkpoint(ablations_raw, ABLATIONS_PATH)

# KB-obj: objects only
if "kb_obj" not in ablations_raw:
    ablations_raw["kb_obj"] = {}
ablations_raw["kb_obj"] = run_ablation_corrector(
    lambda cap, kb: KB_OBJ_PROMPT.format(
        caption=cap,
        obj_facts="\n".join(f"- {f}" for f in kb["hard_facts"]) or "- None",
    ),
    images, captions, knowledge_bases, "KB-obj (objects only)",
    cached=ablations_raw["kb_obj"],
)
save_checkpoint(ablations_raw, ABLATIONS_PATH)

# KB-geom: objects + geometry
if "kb_geom" not in ablations_raw:
    ablations_raw["kb_geom"] = {}
ablations_raw["kb_geom"] = run_ablation_corrector(
    lambda cap, kb: KB_GEOM_PROMPT.format(
        caption=cap,
        obj_facts="\n".join(f"- {f}" for f in kb["hard_facts"]) or "- None",
        spatial_facts="\n".join(f"- {f}" for f in kb["spatial_facts"]) or "- None",
    ),
    images, captions, knowledge_bases, "KB-geom (objects+geometry)",
    cached=ablations_raw["kb_geom"],
)
save_checkpoint(ablations_raw, ABLATIONS_PATH)

# KB-dump: full KB, unstructured rewrite
if "kb_dump" not in ablations_raw:
    ablations_raw["kb_dump"] = {}
ablations_raw["kb_dump"] = run_ablation_corrector(
    lambda cap, kb: KB_DUMP_PROMPT.format(
        caption=cap,
        obj_facts="\n".join(f"- {f}" for f in kb["hard_facts"]) or "- None",
        spatial_facts="\n".join(f"- {f}" for f in kb["spatial_facts"]) or "- None",
        visual_desc=kb["visual_description"][:300] or "- None",
    ),
    images, captions, knowledge_bases, "KB-dump (unstructured)",
    cached=ablations_raw["kb_dump"],
)
save_checkpoint(ablations_raw, ABLATIONS_PATH)

print(f"\nAll ablation variants computed. Saved to Drive.")
for variant, data in ablations_raw.items():
    n_changed = sum(1 for img_id in data if data[img_id] != captions.get(img_id, ""))
    print(f"  {variant}: {n_changed}/{len(data)} captions changed")

In [ ]:
# ============================================================
# Cell 7 — R-POPE Evaluation (LLM-Judge, all methods)
# ============================================================
# Methods evaluated:
#   b1       : Original BLIP-2 captions
#   b2       : Self-refinement
#   b3       : Blind correction
#   kb_obj   : KB objects only
#   kb_geom  : KB objects + geometry
#   kb_dump  : Full KB unstructured
#   relcheck : Full RelCheck v2 (structured + verified)
# Checkpoint: loads from Drive if already computed.

RPOPE_PATH = f"{SAVE_DIR}/rpope_results.json"

RPOPE_PROMPT = """Based ONLY on this caption, answer the question with 'yes' or 'no'.
Do not use any external knowledge. Only use information stated in the caption.

Caption: "{caption}"
Question: {question}

Answer (yes or no):"""


def rpope_judge(caption, question):
    raw = llm_call(
        [{"role": "user", "content": RPOPE_PROMPT.format(caption=caption, question=question)}],
        max_tokens=5,
    )
    if raw:
        raw = raw.lower()
        if "yes" in raw and "no" not in raw:
            return "yes"
        if "no" in raw:
            return "no"
    return "unknown"


def classify_question(question):
    q = question.lower()
    for v in ["playing","holding","eating","sitting","standing","running","walking",
               "riding","driving","reading","carrying","cooking","swimming","flying",
               "climbing","jumping","drinking","pointing","pulling","pushing","cutting",
               "hugging","leaning","lying","reaching","touching","feeding","surfing",
               "skiing","skating","waving","looking","talking","throwing","catching"]:
        if v in q:
            return "ACTION"
    for p in [" on a "," on the "," on top "," in a "," in the "," inside "," outside ",
               " near "," next to "," behind "," above "," below "," under "," beside ",
               " between "," in front of "," across "," along "," around "]:
        if p in f" {q} ":
            return "SPATIAL"
    for kw in ["wearing","color","colour","red","blue","green","yellow","black","white",
                "brown","large","small","big","tall","short","old","young","made of"]:
        if kw in q:
            return "ATTRIBUTE"
    return "OTHER"


def compute_metrics(correct, total):
    """Compute precision/recall/F1/yes_ratio from per-question results."""
    # correct is list of (pred, gt) tuples
    tp = sum(1 for p, g in correct if p == "yes" and g == "yes")
    fp = sum(1 for p, g in correct if p == "yes" and g == "no")
    fn = sum(1 for p, g in correct if p == "no" and g == "yes")
    tn = sum(1 for p, g in correct if p == "no" and g == "no")
    acc = (tp + tn) / total if total else 0
    prec = tp / (tp + fp) if (tp + fp) else 0
    rec = tp / (tp + fn) if (tp + fn) else 0
    f1 = 2 * prec * rec / (prec + rec) if (prec + rec) else 0
    yes_r = (tp + fp) / total if total else 0
    return {"accuracy": acc, "precision": prec, "recall": rec, "f1": f1,
            "yes_ratio": yes_r, "n": total}


# ── Build caption dict per method ──
method_captions = {
    "b1":       captions,
    "b2":       ablations_raw["b2"],
    "b3":       ablations_raw["b3"],
    "kb_obj":   ablations_raw["kb_obj"],
    "kb_geom":  ablations_raw["kb_geom"],
    "kb_dump":  ablations_raw["kb_dump"],
    "relcheck": {img_id: r["corrected"] for img_id, r in enriched.items()},
}

# ── Load or compute R-POPE ──
rpope_raw = load_checkpoint(RPOPE_PATH) or {}
# rpope_raw[img_id][method] = {"pred": "yes"/"no", "gt": ..., "question": ..., "type": ...}

methods_to_eval = [m for m in method_captions if m not in
                   # Check if ALL images already done for this method
                   [m for m in method_captions
                    if all(m in rpope_raw.get(img_id, {}) for img_id in images)]]

if not methods_to_eval:
    print(f"All R-POPE results loaded from Drive.")
else:
    print(f"Running R-POPE for methods: {methods_to_eval}")
    n_imgs = len(images)

    for img_idx, img_id in enumerate(images):
        if img_id not in rpope_raw:
            rpope_raw[img_id] = {}
        questions = img_to_questions.get(img_id, [])

        for entry in questions[:10]:  # cap at 10 q/image to control cost
            q = entry["question"]
            gt = entry["answer"]
            qtype = classify_question(q)

            for method in methods_to_eval:
                key = f"{method}|||{q}"
                if key not in rpope_raw[img_id]:
                    cap = method_captions[method].get(img_id, captions[img_id])
                    pred = rpope_judge(cap, q)
                    rpope_raw[img_id][key] = {"pred": pred, "gt": gt, "type": qtype, "method": method}
                    time.sleep(0.1)

        if (img_idx + 1) % 50 == 0:
            print(f"  [{img_idx+1}/{n_imgs}] evaluated")
            save_checkpoint(rpope_raw, RPOPE_PATH)

    save_checkpoint(rpope_raw, RPOPE_PATH)
    print("Done. R-POPE results saved.")

# ── Aggregate results ──
method_results = {m: [] for m in method_captions}  # list of (pred, gt) per method
method_type_results = {m: defaultdict(list) for m in method_captions}
per_image_results = defaultdict(dict)  # img_id -> method -> (correct, total)
corrected_image_ids = {img_id for img_id, r in enriched.items() if r["status"] == "modified"}

for img_id, img_data in rpope_raw.items():
    if img_id not in images:
        continue
    for key, entry in img_data.items():
        m = entry["method"]
        pred, gt = entry["pred"], entry["gt"]
        qtype = entry["type"]
        if pred != "unknown":
            method_results[m].append((pred, gt))
            method_type_results[m][qtype].append((pred, gt))

# ── Print Table 1: Main R-POPE LLM-Judge Results ──
print(f"\n{'='*75}")
print(f"TABLE 1: R-POPE (LLM-Judge) — {len(images)} images")
print(f"{'='*75}")
print(f"{'Method':<18} {'Accuracy':>10} {'Precision':>10} {'Recall':>9} {'F1':>8} {'Yes%':>7} {'N':>6}")
print("-" * 75)
method_labels = {
    "b1": "B1: No correction", "b2": "B2: Self-refine",
    "b3": "B3: Blind correct", "relcheck": "RelCheck v2",
}
for m in ["b1", "b2", "b3", "relcheck"]:
    if method_results[m]:
        r = compute_metrics(method_results[m], len(method_results[m]))
        label = method_labels.get(m, m)
        print(f"{label:<18} {r['accuracy']:>10.1%} {r['precision']:>10.1%} "
              f"{r['recall']:>9.1%} {r['f1']:>8.1%} {r['yes_ratio']:>7.1%} {r['n']:>6}")

# ── Table 3: KB Source Ablation ──
print(f"\n{'='*75}")
print(f"TABLE 3: KB Source Ablation")
print(f"{'='*75}")
print(f"{'KB Source':<25} {'Accuracy':>10} {'Delta vs B1':>12} {'N':>6}")
print("-" * 55)
b1_acc = compute_metrics(method_results["b1"], len(method_results["b1"]))["accuracy"] if method_results["b1"] else 0
for m, label in [("b3", "No KB (B3)"), ("kb_obj", "Objects only"),
                  ("kb_geom", "Objects + geometry"), ("relcheck", "Full KB (RelCheck v2)")]:
    if method_results[m]:
        r = compute_metrics(method_results[m], len(method_results[m]))
        delta = r['accuracy'] - b1_acc
        print(f"{label:<25} {r['accuracy']:>10.1%} {delta:>+12.1%} {r['n']:>6}")

# ── Table 4: Correction Method Ablation ──
print(f"\n{'='*75}")
print(f"TABLE 4: Correction Method Ablation")
print(f"{'='*75}")
print(f"{'Method':<28} {'Accuracy':>10} {'Delta vs B1':>12} {'N':>6}")
print("-" * 58)
for m, label in [("b3", "Blind (no evidence)"),
                  ("kb_dump", "KB dump (unstructured)"),
                  ("relcheck", "Structured (RelCheck v2)")]:
    if method_results[m]:
        r = compute_metrics(method_results[m], len(method_results[m]))
        delta = r['accuracy'] - b1_acc
        print(f"{label:<28} {r['accuracy']:>10.1%} {delta:>+12.1%} {r['n']:>6}")

# ── Table 7: Filtered R-POPE (corrected images only) ──
print(f"\n{'='*75}")
print(f"TABLE 7: Filtered R-POPE (RelCheck-modified images only, N={len(corrected_image_ids)})")
print(f"{'='*75}")
print(f"{'Method':<18} {'Accuracy':>10} {'Delta':>10} {'N questions':>12}")
print("-" * 52)
for m in ["b1", "relcheck"]:
    filt = [(pred, gt) for img_id in corrected_image_ids
            for key, entry in rpope_raw.get(img_id, {}).items()
            if entry.get("method") == m and entry["pred"] != "unknown"
            for pred, gt in [(entry["pred"], entry["gt"])]]
    if filt:
        r = compute_metrics(filt, len(filt))
        delta_str = ""
        if m == "relcheck":
            b1_filt = [(pred, gt) for img_id in corrected_image_ids
                       for key, entry in rpope_raw.get(img_id, {}).items()
                       if entry.get("method") == "b1" and entry["pred"] != "unknown"
                       for pred, gt in [(entry["pred"], entry["gt"])]]
            if b1_filt:
                b1r = compute_metrics(b1_filt, len(b1_filt))
                delta_str = f"{r['accuracy']-b1r['accuracy']:>+10.1%}"
        print(f"{m:<18} {r['accuracy']:>10.1%} {delta_str:>10} {r['n']:>12}")

# ── Table 8: Per-relation-type breakdown ──
print(f"\n{'='*75}")
print(f"TABLE 8: Per-Relation-Type Breakdown (B1 vs RelCheck v2)")
print(f"{'='*75}")
print(f"{'Type':<12} {'B1 Acc':>8} {'RC Acc':>8} {'Delta':>8} {'N':>6}")
print("-" * 44)
for qtype in ["SPATIAL", "ACTION", "ATTRIBUTE", "OTHER"]:
    b1_t = method_type_results["b1"][qtype]
    rc_t = method_type_results["relcheck"][qtype]
    if b1_t and rc_t:
        b1r = compute_metrics(b1_t, len(b1_t))
        rcr = compute_metrics(rc_t, len(rc_t))
        delta = rcr['accuracy'] - b1r['accuracy']
        print(f"{qtype:<12} {b1r['accuracy']:>8.1%} {rcr['accuracy']:>8.1%} {delta:>+8.1%} {len(b1_t):>6}")

In [ ]:
# ============================================================
# Cell 8 — CLIPScore (Table 5)
# ============================================================
# Measures image-caption alignment for each method.
# Uses OpenCLIP ViT-B/32 — loads locally on Colab GPU.

import open_clip

print("Loading CLIP model...")
clip_model, _, clip_preprocess = open_clip.create_model_and_transforms(
    'ViT-B-32', pretrained='openai'
)
clip_tokenizer = open_clip.get_tokenizer('ViT-B-32')
clip_model = clip_model.to(DEVICE).eval()
print("CLIP loaded.")


@torch.no_grad()
def clip_score(image, caption):
    """Compute cosine similarity between image and caption in CLIP space."""
    img_tensor = clip_preprocess(image).unsqueeze(0).to(DEVICE)
    txt_tensor = clip_tokenizer([caption]).to(DEVICE)
    img_feat = clip_model.encode_image(img_tensor)
    txt_feat = clip_model.encode_text(txt_tensor)
    img_feat /= img_feat.norm(dim=-1, keepdim=True)
    txt_feat /= txt_feat.norm(dim=-1, keepdim=True)
    return (img_feat * txt_feat).sum().item()


CLIP_PATH = f"{SAVE_DIR}/clip_scores.json"
clip_raw = load_checkpoint(CLIP_PATH) or {}

methods_clip = ["b1", "b2", "b3", "relcheck"]
todo_clip = [img_id for img_id in images
             if not all(m in clip_raw.get(img_id, {}) for m in methods_clip)]

if not todo_clip:
    print(f"Loaded {len(clip_raw)} cached CLIPScores.")
else:
    print(f"Computing CLIPScore for {len(todo_clip)} images...")
    for idx, img_id in enumerate(todo_clip):
        if img_id not in clip_raw:
            clip_raw[img_id] = {}
        img = images[img_id]
        for m in methods_clip:
            if m not in clip_raw[img_id]:
                cap = method_captions[m].get(img_id, captions[img_id])
                clip_raw[img_id][m] = clip_score(img, cap)
        if (idx + 1) % 100 == 0:
            print(f"  [{idx+1}/{len(todo_clip)}]")
            save_checkpoint(clip_raw, CLIP_PATH)
    save_checkpoint(clip_raw, CLIP_PATH)
    print("Done. CLIPScores saved.")

# ── Table 5: CLIPScore ──
print(f"\n{'='*55}")
print(f"TABLE 5: CLIPScore (image-caption alignment)")
print(f"{'='*55}")
print(f"{'Method':<22} {'Mean CLIPScore':>14} {'Delta vs B1':>12}")
print("-" * 50)

b1_scores = [clip_raw[img_id]["b1"] for img_id in clip_raw if "b1" in clip_raw[img_id]]
b1_mean = np.mean(b1_scores) if b1_scores else 0

for m, label in [("b1","B1: No correction"), ("b2","B2: Self-refine"),
                  ("b3","B3: Blind"), ("relcheck","RelCheck v2")]:
    scores = [clip_raw[img_id][m] for img_id in clip_raw if m in clip_raw.get(img_id, {})]
    if scores:
        mean_s = np.mean(scores)
        delta = mean_s - b1_mean
        print(f"{label:<22} {mean_s:>14.4f} {delta:>+12.4f}")

In [ ]:
# ============================================================
# Cell 9 — Table 9: Pipeline Statistics + Save All CSVs
# ============================================================

# ── Table 9: Pipeline Stats ──
print(f"{'='*55}")
print(f"TABLE 9: Pipeline Statistics")
print(f"{'='*55}")

total_images = len(images)
n_modified = sum(1 for r in enriched.values() if r["status"] == "modified")
n_errors_total = sum(len(r["errors"]) for r in enriched.values())
n_missing_total = sum(len(r["missing"]) for r in enriched.values())
avg_edit = np.mean([r["edit_rate"] for r in enriched.values() if r["status"] == "modified"]) if n_modified else 0

total_questions = sum(len(qs) for qs in img_to_questions.values())
avg_q = total_questions / total_images

all_obj_counts = [len(Counter(l for l,_,_ in kb["detections"])) for kb in knowledge_bases.values()]
all_spatial_counts = [len(kb["spatial_facts"]) for kb in knowledge_bases.values()]

print(f"  Total images processed:           {total_images}")
print(f"  R-Bench questions:                {total_questions} ({avg_q:.1f}/image)")
print(f"  Images with errors/missing found: {sum(1 for r in enriched.values() if r['errors'] or r['missing'])}")
print(f"  Images modified (passed verify):  {n_modified} ({n_modified/total_images:.0%})")
print(f"  Total errors corrected:           {n_errors_total}")
print(f"  Total missing facts added:        {n_missing_total}")
print(f"  Avg edit rate (modified images):  {avg_edit:.0%}")
print(f"  Avg objects detected/image:       {np.mean(all_obj_counts):.1f}")
print(f"  Avg spatial facts/image:          {np.mean(all_spatial_counts):.1f}")

# ── Save all CSVs to Drive ──
csv_dir = f"{SAVE_DIR}/csvs"
os.makedirs(csv_dir, exist_ok=True)

# Per-image results CSV
with open(f"{csv_dir}/per_image_results.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=[
        "img_id", "original_caption", "relcheck_caption",
        "n_errors", "n_missing", "edit_rate", "status",
        "n_objects", "n_spatial_facts",
    ])
    writer.writeheader()
    for img_id in images:
        r = enriched.get(img_id, {})
        kb = knowledge_bases.get(img_id, {})
        writer.writerow({
            "img_id": img_id,
            "original_caption": r.get("caption", ""),
            "relcheck_caption": r.get("corrected", ""),
            "n_errors": len(r.get("errors", [])),
            "n_missing": len(r.get("missing", [])),
            "edit_rate": f"{r.get('edit_rate', 0):.3f}",
            "status": r.get("status", ""),
            "n_objects": len(Counter(l for l,_,_ in kb.get("detections", []))),
            "n_spatial_facts": len(kb.get("spatial_facts", [])),
        })

# R-POPE summary CSV
with open(f"{csv_dir}/rpope_summary.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["method","accuracy","precision","recall","f1","yes_ratio","n"])
    writer.writeheader()
    for m in ["b1","b2","b3","kb_obj","kb_geom","kb_dump","relcheck"]:
        if method_results[m]:
            r = compute_metrics(method_results[m], len(method_results[m]))
            writer.writerow({"method": m, **{k: f"{v:.4f}" for k, v in r.items()}})

# Per-relation-type CSV
with open(f"{csv_dir}/per_type_results.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["method","type","accuracy","n"])
    writer.writeheader()
    for m in ["b1","relcheck"]:
        for qtype in ["SPATIAL","ACTION","ATTRIBUTE","OTHER"]:
            data = method_type_results[m][qtype]
            if data:
                r = compute_metrics(data, len(data))
                writer.writerow({"method": m, "type": qtype,
                                  "accuracy": f"{r['accuracy']:.4f}", "n": r["n"]})

print(f"\nCSVs saved to {csv_dir}/")
print(f"  per_image_results.csv")
print(f"  rpope_summary.csv")
print(f"  per_type_results.csv")

In [ ]:
# ============================================================
# Cell 10 — Figures (saved to Drive)
# ============================================================

COLORS = {
    "b1": "#aec6cf", "b2": "#b5d5c5", "b3": "#ffd966",
    "kb_obj": "#f4a261", "kb_geom": "#e76f51",
    "kb_dump": "#9b72cf", "relcheck": "#2a9d8f"
}

def method_acc(m):
    d = method_results.get(m, [])
    return compute_metrics(d, len(d))["accuracy"] if d else 0


# ── Figure 4: Per-Relation-Type Grouped Bar Chart ──
fig, ax = plt.subplots(figsize=(9, 5))
qtypes = ["SPATIAL", "ACTION", "ATTRIBUTE", "OTHER"]
x = np.arange(len(qtypes))
width = 0.35

b1_accs = [compute_metrics(method_type_results["b1"][t], len(method_type_results["b1"][t]))["accuracy"]
           if method_type_results["b1"][t] else 0 for t in qtypes]
rc_accs = [compute_metrics(method_type_results["relcheck"][t], len(method_type_results["relcheck"][t]))["accuracy"]
           if method_type_results["relcheck"][t] else 0 for t in qtypes]

bars1 = ax.bar(x - width/2, b1_accs, width, label="B1: Original", color=COLORS["b1"], edgecolor="gray")
bars2 = ax.bar(x + width/2, rc_accs, width, label="RelCheck v2", color=COLORS["relcheck"], edgecolor="gray")

ax.set_xlabel("Relation Type", fontsize=12)
ax.set_ylabel("R-POPE Accuracy (LLM-Judge)", fontsize=12)
ax.set_title("Figure 4: Per-Relation-Type Accuracy (B1 vs RelCheck v2)", fontsize=13)
ax.set_xticks(x)
ax.set_xticklabels(qtypes)
ax.set_ylim(0, 1.0)
ax.legend()
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))

for bars in [bars1, bars2]:
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, h + 0.01, f"{h:.0%}",
                ha='center', va='bottom', fontsize=9)

plt.tight_layout()
fig4_path = f"{FIGS_DIR}/fig4_per_relation_type.png"
plt.savefig(fig4_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"Figure 4 saved: {fig4_path}")


# ── Figure 5: KB Source Ablation Bar Chart ──
fig, ax = plt.subplots(figsize=(9, 5))
abl_methods = ["b3", "kb_obj", "kb_geom", "relcheck"]
abl_labels = ["No KB\n(B3)", "Objects\nOnly", "Objects +\nGeometry", "Full KB\n(RelCheck v2)"]
abl_accs = [method_acc(m) for m in abl_methods]
abl_colors = [COLORS[m] for m in abl_methods]

bars = ax.bar(abl_labels, abl_accs, color=abl_colors, edgecolor="gray", width=0.5)
ax.axhline(y=method_acc("b1"), color="gray", linestyle="--", linewidth=1.5, label="B1 baseline")
ax.set_ylabel("R-POPE Accuracy (LLM-Judge)", fontsize=12)
ax.set_title("Figure 5: KB Source Ablation", fontsize=13)
ax.set_ylim(max(0, min(abl_accs) - 0.05), min(1.0, max(abl_accs) + 0.05))
ax.legend()
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))

for bar, acc in zip(bars, abl_accs):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
            f"{acc:.1%}", ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.tight_layout()
fig5_path = f"{FIGS_DIR}/fig5_kb_source_ablation.png"
plt.savefig(fig5_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"Figure 5 saved: {fig5_path}")


# ── Figure 6: Correction Method Ablation ──
fig, ax = plt.subplots(figsize=(8, 5))
corr_methods = ["b3", "kb_dump", "relcheck"]
corr_labels = ["Blind\n(No evidence)", "KB Dump\n(Unstructured)", "Structured\n(RelCheck v2)"]
corr_accs = [method_acc(m) for m in corr_methods]
corr_colors = [COLORS[m] for m in corr_methods]

bars = ax.bar(corr_labels, corr_accs, color=corr_colors, edgecolor="gray", width=0.4)
ax.axhline(y=method_acc("b1"), color="gray", linestyle="--", linewidth=1.5, label="B1 baseline")
ax.set_ylabel("R-POPE Accuracy (LLM-Judge)", fontsize=12)
ax.set_title("Figure 6: Correction Method Ablation", fontsize=13)
ax.set_ylim(max(0, min(corr_accs) - 0.05), min(1.0, max(corr_accs) + 0.05))
ax.legend()
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))

for bar, acc in zip(bars, corr_accs):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
            f"{acc:.1%}", ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.tight_layout()
fig6_path = f"{FIGS_DIR}/fig6_correction_method.png"
plt.savefig(fig6_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"Figure 6 saved: {fig6_path}")


# ── Final summary ──
print(f"\n{'='*60}")
print(f"FULL RUN COMPLETE — {len(images)} images")
print(f"{'='*60}")
print(f"  B1 accuracy:        {method_acc('b1'):.1%}")
print(f"  RelCheck accuracy:  {method_acc('relcheck'):.1%}")
print(f"  Delta:              {method_acc('relcheck') - method_acc('b1'):+.1%}")
print(f"  Images modified:    {n_modified}/{total_images} ({n_modified/total_images:.0%})")
print(f"\nAll outputs in: {SAVE_DIR}/")
print(f"  csvs/          — Tables 1,3,4,5,7,8,9 as CSV")
print(f"  figures/       — Figures 4,5,6 as PNG")
print(f"  *.json         — Raw results for further analysis")